In [5]:
pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_core.documents import Document

In [11]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source":"https://www.google.com"}
)

In [12]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [13]:
type(sample_doc)

langchain_core.documents.base.Document

In [14]:
# Text data -> document
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Python.txt", encoding="utf-8")

In [15]:
document = loader.load()

In [16]:
document

[Document(metadata={'source': 'Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.\n

In [10]:
# pdf data document
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("research.pdf")

# document = pdf_loader.load()
# document

In [12]:
# pdf data document

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("research.pdf")

# document = pdf_loader.load()
# document

## Ingestion Pipeline

In [17]:
# Data => documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [18]:
def load_all_pdfs():
    folder_path = "pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1
            
    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [19]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [20]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [21]:
## chunks
# !pip install langchain_text_splitters

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [23]:
chunks = split_docs(all_pdf_documents)

In [24]:
len(chunks)

321

In [25]:
#chunks

### Embedding

In [26]:
from sentence_transformers import SentenceTransformer

In [27]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("Loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())

    
    def generate_embeddings(self,text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [28]:
embedding_manager = EmbeddingManager()

Loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\aman upadhyay\AppData\Local\Temp\ipykernel_24772\1985050592.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


### Vector Store

In [29]:
import chromadb
import uuid

In [47]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description:":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )
            
        print("total documents added in vector store", len(documents_content))
        print("docs in collection:", self.collection.count())

In [48]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [49]:
# chunks   # chunk ==> page_content, metadata   # every chunk is an individual document in itself

In [50]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector store 321
docs in collection: 321


## Retrieval Pipeline

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
class RAGRetreiver:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip( ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs
            

In [56]:
rag_retriever = RAGRetreiver(embedding_manager, vector_store)

In [58]:
rag_retriever.retrieve("What is encoder decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_863a9ef4-a2f2-4f40-8fd5-339d09b50307',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'type': 'Conference Proceedings',
   'creationdate': '2026-07-25T20:07:29+00:00',
   'content_length': 447,
   'published': '2017',
   'language': 'en-US',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'created': '2017',
   'producer': 'pdfcpu v0.12.1 dev',
   'publisher': 'Curran Associates, Inc.'

## Integrate with LLM

### OpenAI-GPT

In [ ]:
API_KEY_OPENAI = ""

In [ ]:
# !pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4",
    temperature = 0.1, # degree of creativity  # low value when fact based and high value when need creativity (>0.7)
    max_tokens = 1024
)

In [ ]:
# generate our retrieval-agumented output

def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f"""use the given context to generate the answer for the query
                Context:{context}
                Query:{query}"""

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG", rag_retriever, llm)

In [ ]:
print(answer)

### Groq

In [ ]:
API_Key_GROQ=""

In [ ]:
!pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "qwen/qwen3-32b",
    temperature = 0.1, # degree of creativity  # low value when fact based and high value when need creativity (>0.7)
    max_tokens = 1024
)

In [ ]:
# generate our retrieval-agumented output

def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f"""use the given context to generate the answer for the query
                Context:{context}
                Query:{query}"""

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG", rag_retriever, llm)

In [ ]:
print(answer)